<font color="steelblue" size = 20px>***📘机器学习笔记***</font>

# <font color="limegreen">一.多特征线性回归模型</font>

In [ ]:
#将要使用到的库
import copy
import math
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "SimSun"] #使得绘制图支持中文

## <font color="gold">1.自定义实验数据</font>

<img src="images\Data Correspondence.jpg" width="400" />

In [ ]:
X_train = np.array([[2104, 5, 1, 45], [1416, 3, 2, 40], [852, 2, 1, 35]])
y_train = np.array([460, 232, 178])

## <font color="gold">2.数据归一化</font>

数据标准化原理：  
<img src="images\Normalization.jpg" width="350" />

在数据z-score归一化中：
使用numpy库中的  
mean方法<font color="orange">X_mean = np.mean(X_train, axis=0)</font>来计算特征值的数学期望，其中axis=0代表按列计算  
以及std方法<font color="orange">X_std = np.std(X_train, axis=0)</font>来计算特征值的标准差，其中axis=0同样代表按列计算  
最后对特征进行标准化<font color="orange">X_train_norm = (X_train - X_mean) / X_std</font>，此时的数据数学期望为0，标准差为1

In [ ]:
print("数据归一化前：")
print(X_train)

print("数据归一化后：")
X_mean = np.mean(X_train, axis=0)
X_std = np.std(X_train, axis=0)
X_train_norm = (X_train - X_mean) / X_std
print(X_train_norm)

## <font color="gold">3.开始定义机器学习函数</font>

- 计算成本函数  
成本函数公式：  
<img src="images\CostFunction.png" width="300" />  
其中：  
<img src="images\Ffunction.png" width="250" />

In [ ]:
def compute_cost(X, y, w, b):
    # X : 数据，m个样本，每个样本有n个特征
    # y : 目标值
    # w : 模型参数
    # b : 模型参数
    m = X.shape[0] #获取训练数据数量，此时m = 3
    cost = 0.0

    for i in range(m):
        f_wb_i = np.dot(X[i] , w) + b #使用numpy加速向量计算
        cost = cost + (f_wb_i - y[i])**2
    cost /= (2*m)

    return cost

- 计算梯度  
梯度计算公式（以线性回归为例）：  
<img src="images\DescentFunction.png" width="300" />  
其中：  
<img src="images\Ffunction.png" width="250" />

In [ ]:
def compute_gradient(X, y, w, b):
    # X : 数据，m个样本，每个样本有n个特征
    # y : 目标值
    # w : 模型参数
    # b : 模型参数
    m,n = X.shape #获取训练数据的数据数量m，以及单个数据里有多少特征值n
    dj_dw = np.zeros((n,))
    dj_db = 0.

    for i in range (m):
        err = (np.dot(X[i], w) + b) - y[i]
        for j in range(n):
            dj_dw[j] = dj_dw[j] + err * X[i,j]
        dj_db += err
    dj_dw /= m
    dj_db /= m

    return dj_db,dj_dw

- 开始根据梯度更新每个参数  
<img src="images\UpdateParameter.png" width="400" />   
注意：  
1.其中的α为学习率，决定参数的改变幅度  
2.两个参数要同步更新

In [ ]:
def gradient_descent(X, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters):
    # X : 数据，m个样本，每个样本有n个特征
    # y : 目标值
    # w_in : 模型训练参数的初始值
    # b_in : 模型训练参数的初始值
    # cost_function : 成本计算函数
    # gradient_function : 梯度计算函数
    # alpha : α学习率
    # num_iters : 学习次数
    
    J_history = [] #用来记录参数更细的历史

    w = copy.deepcopy(w_in) #创建一个和w_in不相关但是数值相同的变量
    b = b_in

    for i in range(num_iters):
        dj_db,dj_dw = gradient_function(X, y, w, b)

        w = w - alpha * dj_dw
        b = b - alpha * dj_db

        if i < 100000:
            J_history.append( cost_function(X, y, w, b))

        if i % math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:4d}: Cost {J_history[-1]:8.2f}")

    return w, b, J_history

## <font color="gold">4.开始进行机器学习</font>

定义机器学习参数  
开始学习前的最后准备

In [ ]:
initial_w = np.zeros(4) #初始化线性回归w参数，全部为0
initial_b = 0 #初始化线性回归b参数为0

iterations =  1000 #学习的次数
alpha = 0.01 #学习率

开始机器学习

In [ ]:
w_final, b_final, J_hist = gradient_descent(X_train_norm, y_train, initial_w, initial_b, compute_cost, compute_gradient, alpha, iterations)

绘制图，观察梯度下降过程中成本函数J的变化

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, constrained_layout = True, figsize = (10 , 4))
#创建画布和子图，创建了一行二列的画图布局，其中fig是画布，ax1和ax2是左右两个子图，整个画布宽12，高4，布局为自动调整子图间距，防止标题、标签互相遮挡

ax1.plot(J_hist) #传入训练历史过程
ax2.plot(900 + np.arange(len(J_hist[900:])),J_hist[900:])

ax1.set_title("成本随训练次数变化");  ax2.set_title("最后100次训练成本的变化")
ax1.set_ylabel('Cost')             ;  ax2.set_ylabel('Cost') 
ax1.set_xlabel('iteration step')   ;  ax2.set_xlabel('iteration step') 

plt.show()

# <font color="limegreen">二.逻辑回归模型</font>